## Logistic Regression: how it works and why it suits this problem
Logistic Regression learns a weight for each feature, combines them into a score, and passes the score through a sigmoid function to give the probability that a session ends in a purchase. A session is predicted as "purchase" when that probability passes a threshold (0.5 by default).

It suits this problem because (1) the target is binary (purchase or no purchase), (2) it is fast and stable on about 10,000 rows, (3) its coefficients show which behaviours push purchase likelihood up or down, which makes it the group's interpretable (white-box) baseline, and (4) it supports class weighting, which matters because only 15.5% of sessions are purchases. Its limitation is that it is linear, so it cannot capture feature interactions unless they are engineered.

In [1]:
import pandas as pd

# Feature-selected data (Member 3): 18 columns
X_train = pd.read_csv("../data/processed/X_train_selected.csv")
X_test = pd.read_csv("../data/processed/X_test_selected.csv")

# Engineered data BEFORE selection (21 columns) - used only for the Cell 7 comparison
X_train_eng = pd.read_csv("../data/processed/X_train_engineered.csv")

y_train = pd.read_csv("../data/y_train.csv").squeeze("columns").astype(int)
# Test set stays untouched until the GROUP's final model is selected
y_test = pd.read_csv("../data/y_test.csv").squeeze("columns").astype(int)

assert len(X_train) == len(y_train) == len(X_train_eng)
assert len(X_test) == len(y_test)
assert list(X_train.columns) == list(X_test.columns), "Train/test column mismatch"

print("Revenue in X_train:", "Revenue" in X_train.columns)
print("Revenue in X_test:", "Revenue" in X_test.columns)
print("Training features (selected):", X_train.shape)
print("Training features (engineered, pre-selection):", X_train_eng.shape)
print("Test features (selected):", X_test.shape)
print("Dropped by feature selection:", sorted(set(X_train_eng.columns) - set(X_train.columns)))

Revenue in X_train: False
Revenue in X_test: False
Training features (selected): (9764, 18)
Training features (engineered, pre-selection): (9764, 21)
Test features (selected): (2441, 18)
Dropped by feature selection: ['ProductRelated', 'ProductRelated_Duration', 'Region']


In [2]:
import sys
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.preprocessing.encoding_scaling import CATEGORICAL_COLUMNS, build_preprocessor

categorical_cols = [col for col in CATEGORICAL_COLUMNS if col in X_train.columns]
numerical_cols = [col for col in X_train.columns if col not in categorical_cols]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
}

print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)

Categorical columns: ['Month', 'OperatingSystems', 'Browser', 'TrafficType', 'VisitorType', 'Weekend', 'VisitorType_Weekend']
Numerical columns: ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'TotalPages', 'TotalDuration', 'AvgTimePerPage']


In [3]:
# Plain Logistic Regression — no class weighting, no tuning yet.
# This is the first checkpoint, not just a step on the way to tuning.
baseline_pipeline = Pipeline([
    ("preprocessing", build_preprocessor(numerical_cols, categorical_cols)),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])

baseline_scores = cross_validate(
    baseline_pipeline, X_train, y_train,
    scoring=scoring, cv=cv, n_jobs=-1,
)

baseline_results = pd.DataFrame({
    "Metric": list(scoring.keys()),
    "Mean": [baseline_scores[f"test_{m}"].mean() for m in scoring],
    "Std": [baseline_scores[f"test_{m}"].std() for m in scoring],
})

print("Logistic Regression — untuned, unweighted baseline")
baseline_results

Logistic Regression — untuned, unweighted baseline


,Metric,Mean,Std
0,f1,0.615856,0.013985
1,precision,0.705749,0.024017
2,recall,0.547168,0.022252
3,roc_auc,0.912431,0.007365
4,average_precision,0.686490,0.018986


In [4]:
from sklearn.dummy import DummyClassifier

# "Always predict no purchase" — demonstrates why accuracy alone is
# misleading, given the dataset is ~84.5% no-purchase / 15.5% purchase
majority_pipeline = Pipeline([
    ("preprocessing", build_preprocessor(numerical_cols, categorical_cols)),
    ("model", DummyClassifier(strategy="most_frequent")),
])

majority_scores = cross_validate(
    majority_pipeline, X_train, y_train,
    scoring=scoring, cv=cv, n_jobs=-1,
)

majority_results = pd.DataFrame({
    "Metric": list(scoring.keys()),
    "Mean": [majority_scores[f"test_{m}"].mean() for m in scoring],
    "Std": [majority_scores[f"test_{m}"].std() for m in scoring],
})

print("Always-predict-no-purchase baseline")
majority_results

Always-predict-no-purchase baseline


,Metric,Mean,Std
0,f1,0.000000,0.000000
1,precision,0.000000,0.000000
2,recall,0.000000,0.000000
3,roc_auc,0.500000,0.000000
4,average_precision,0.156288,0.000199


In [5]:
# Only the weighting changes here — same features, same CV, same
# preprocessing — so any difference can be attributed to weighting alone
weighted_pipeline = Pipeline([
    ("preprocessing", build_preprocessor(numerical_cols, categorical_cols)),
    ("model", LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")),
])

weighted_scores = cross_validate(
    weighted_pipeline, X_train, y_train,
    scoring=scoring, cv=cv, n_jobs=-1,
)

weighting_comparison = pd.DataFrame({
    "Metric": list(scoring.keys()),
    "Mean_baseline": [baseline_scores[f"test_{m}"].mean() for m in scoring],
    "Std_baseline": [baseline_scores[f"test_{m}"].std() for m in scoring],
    "Mean_weighted": [weighted_scores[f"test_{m}"].mean() for m in scoring],
    "Std_weighted": [weighted_scores[f"test_{m}"].std() for m in scoring],
})

weighting_comparison

,Metric,Mean_baseline,Std_baseline,Mean_weighted,Std_weighted
0,f1,0.615856,0.013985,0.647738,0.013602
1,precision,0.705749,0.024017,0.544258,0.017908
2,recall,0.547168,0.022252,0.800782,0.021224
3,roc_auc,0.912431,0.007365,0.916653,0.007847
4,average_precision,0.686490,0.018986,0.666333,0.020980


In [6]:
# Decide which weighting actually won on F1 
baseline_f1 = baseline_scores["test_f1"].mean()
weighted_f1 = weighted_scores["test_f1"].mean()

if weighted_f1 > baseline_f1:
    best_weighting_pipeline = weighted_pipeline
    selected_weighting = "balanced"
    selected_class_weight = "balanced"
else:
    best_weighting_pipeline = baseline_pipeline
    selected_weighting = "unweighted"
    selected_class_weight = None

print("Selected weighting:", selected_weighting)

import numpy as np
from sklearn.model_selection import GridSearchCV

# Tune C (regularization strength) using the SAME CV folds and F1 as
# the refit metric, on whichever weighting was selected above
parameter_grid = {
    "model__C": np.logspace(-3, 2, 12),  # 0.001 to 100
}

search = GridSearchCV(
    estimator=best_weighting_pipeline,
    param_grid=parameter_grid,
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=-1,
    error_score="raise",
)
search.fit(X_train, y_train)

selected_C = search.best_params_["model__C"]
best_index = search.best_index_

# Compare tuned vs untuned using the SAME selected weighting on both
# sides — otherwise the comparison wouldn't isolate tuning's effect
selected_untuned_scores = weighted_scores if selected_weighting == "balanced" else baseline_scores

tuning_comparison = pd.DataFrame({
    "Metric": list(scoring.keys()),
    "Mean_untuned": [selected_untuned_scores[f"test_{m}"].mean() for m in scoring],
    "Std_untuned": [selected_untuned_scores[f"test_{m}"].std() for m in scoring],
    "Mean_tuned": [search.cv_results_[f"mean_test_{m}"][best_index] for m in scoring],
    "Std_tuned": [search.cv_results_[f"std_test_{m}"][best_index] for m in scoring],
})

print("Selected C:", selected_C)
tuning_comparison

Selected weighting: balanced
Selected C: 0.002848035868435802


,Metric,Mean_untuned,Std_untuned,Mean_tuned,Std_tuned
0,f1,0.647738,0.013602,0.666432,0.013995
1,precision,0.544258,0.017908,0.577806,0.016160
2,recall,0.800782,0.021224,0.788323,0.029812
3,roc_auc,0.916653,0.007847,0.911714,0.007209
4,average_precision,0.666333,0.020980,0.663717,0.017478


In [7]:
import numpy as np
from sklearn.model_selection import GridSearchCV

low_grid = {"model__C": np.logspace(-4, -2, 9)}  # 0.0001 to 0.01
low_search = GridSearchCV(
    estimator=best_weighting_pipeline, param_grid=low_grid,
    scoring=scoring, refit="f1", cv=cv, n_jobs=-1,
).fit(X_train, y_train)

print("Best C:", low_search.best_params_["model__C"])
print("Best F1:", low_search.cv_results_["mean_test_f1"][low_search.best_index_])

Best C: 0.0017782794100389228
Best F1: 0.6674717001001523


In [8]:
# Same weighting and C on all conditions, so only the input columns differ.
# NOTE: in the 14-column condition, removing the engineered columns also removes
# the only product-page information (ProductRelated* were dropped by feature selection).
engineered_columns = ["TotalPages", "TotalDuration", "AvgTimePerPage", "VisitorType_Weekend"]

feature_set_conditions = {
    "Engineered, before selection (21)": X_train_eng,
    "Selected by Member 3 (18)": X_train,
    "Selected, 4 engineered columns removed (14)": X_train.drop(columns=engineered_columns),
    "Original raw columns only (17)": X_train_eng.drop(columns=engineered_columns),
}

feature_rows = []
for condition, features in feature_set_conditions.items():
    cat_cols = [c for c in CATEGORICAL_COLUMNS if c in features.columns]
    num_cols = [c for c in features.columns if c not in cat_cols]

    pipeline = Pipeline([
        ("preprocessing", build_preprocessor(num_cols, cat_cols)),
        ("model", LogisticRegression(max_iter=1000, random_state=42,
                                     class_weight=selected_class_weight, C=selected_C)),
    ])
    results = cross_validate(pipeline, features, y_train, scoring=scoring, cv=cv, n_jobs=-1)

    for metric in scoring:
        feature_rows.append({
            "Condition": condition, "Metric": metric,
            "Mean": results[f"test_{metric}"].mean(),
            "Std": results[f"test_{metric}"].std(),
        })

feature_set_results = pd.DataFrame(feature_rows)
feature_set_results.pivot(index="Metric", columns="Condition", values="Mean")

Condition,"Engineered, before selection (21)",Original raw columns only (17),Selected by Member 3 (18),"Selected, 4 engineered columns removed (14)"
Metric,,,,
average_precision,0.662659,0.661582,0.663717,0.664897
f1,0.666565,0.667280,0.666432,0.664505
precision,0.576566,0.580158,0.577806,0.581441
recall,0.790942,0.786356,0.788323,0.776522
roc_auc,0.910716,0.911233,0.911714,0.914790


In [9]:
pagevalues_rows = []
for condition, features in [
    ("With PageValues", X_train),
    ("Without PageValues", X_train.drop(columns=["PageValues"])),
]:
    cat_cols = [c for c in CATEGORICAL_COLUMNS if c in features.columns]
    num_cols = [c for c in features.columns if c not in cat_cols]

    pipeline = Pipeline([
        ("preprocessing", build_preprocessor(num_cols, cat_cols)),
        ("model", LogisticRegression(max_iter=1000, random_state=42,
                                     class_weight=selected_class_weight, C=selected_C)),
    ])
    results = cross_validate(pipeline, features, y_train, scoring=scoring, cv=cv, n_jobs=-1)

    for metric in scoring:
        pagevalues_rows.append({
            "Condition": condition, "Metric": metric,
            "Mean": results[f"test_{metric}"].mean(),
            "Std": results[f"test_{metric}"].std(),
        })

pagevalues_results = pd.DataFrame(pagevalues_rows)
pagevalues_results.pivot(index="Metric", columns="Condition", values="Mean")

Condition,With PageValues,Without PageValues
Metric,,
average_precision,0.663717,0.327624
f1,0.666432,0.384929
precision,0.577806,0.258117
recall,0.788323,0.756865
roc_auc,0.911714,0.742069


## PageValues: availability and leakage
Removing PageValues drops F1 from 0.666 to 0.385 and ROC-AUC from 0.912 to 0.742, so the model depends heavily on it. The dataset authors describe PageValues as the average value of pages a visitor saw before completing a transaction. The documentation does not say whether the current session's own purchase contributes to that value. We therefore cannot establish from the data alone that PageValues is leakage-free or available at prediction time. We treat this as an assumption: in a deployed system PageValues would have to be available before the purchase decision, otherwise real performance would be closer to the "without PageValues" result. The comparison above shows how important the feature is, but it cannot prove it is safe to use.

In [10]:
experiment_summary = pd.DataFrame([
    {"Experiment": "Baseline (unweighted, untuned)", "F1": baseline_scores["test_f1"].mean(), "F1_std": baseline_scores["test_f1"].std()},
    {"Experiment": "Weighted (class_weight='balanced')", "F1": weighted_scores["test_f1"].mean(), "F1_std": weighted_scores["test_f1"].std()},
    {"Experiment": f"Tuned (selected: {selected_weighting}, C={selected_C:.5f})", "F1": search.cv_results_["mean_test_f1"][best_index], "F1_std": search.cv_results_["std_test_f1"][best_index]},
])
experiment_summary

,Experiment,F1,F1_std
0,"Baseline (unweighted, untuned)",0.615856,0.013985
1,Weighted (class_weight='balanced'),0.647738,0.013602
2,"Tuned (selected: balanced, C=0.00285)",0.666432,0.013995


In [11]:
# Stage 6 requirement: every metric as mean +/- standard deviation
def fmt(scores, m):
    return f"{scores['test_' + m].mean():.3f} ± {scores['test_' + m].std():.3f}"

mean_std_table = pd.DataFrame({
    "Metric": list(scoring.keys()),
    "Always no-purchase": [fmt(majority_scores, m) for m in scoring],
    "LR baseline (unweighted)": [fmt(baseline_scores, m) for m in scoring],
    "LR balanced": [fmt(weighted_scores, m) for m in scoring],
    f"LR tuned ({selected_weighting}, C)": [
        f"{search.cv_results_[f'mean_test_{m}'][best_index]:.3f} ± {search.cv_results_[f'std_test_{m}'][best_index]:.3f}"
        for m in scoring
    ],
})
mean_std_table

,Metric,Always no-purchase,LR baseline (unweighted),LR balanced,"LR tuned (balanced, C)"
0,f1,0.000 ± 0.000,0.616 ± 0.014,0.648 ± 0.014,0.666 ± 0.014
1,precision,0.000 ± 0.000,0.706 ± 0.024,0.544 ± 0.018,0.578 ± 0.016
2,recall,0.000 ± 0.000,0.547 ± 0.022,0.801 ± 0.021,0.788 ± 0.030
3,roc_auc,0.500 ± 0.000,0.912 ± 0.007,0.917 ± 0.008,0.912 ± 0.007
4,average_precision,0.156 ± 0.000,0.686 ± 0.019,0.666 ± 0.021,0.664 ± 0.017


In [12]:
# Largest coefficients of the tuned model (trained on training data only)
best_model = search.best_estimator_
feature_names = best_model.named_steps["preprocessing"].get_feature_names_out()
coefficients = best_model.named_steps["model"].coef_[0]

coef_table = pd.DataFrame({"feature": feature_names, "coefficient": coefficients})
coef_table["abs_coefficient"] = coef_table["coefficient"].abs()
coef_table = coef_table.sort_values("abs_coefficient", ascending=False, ignore_index=True)
coef_table.head(10)

,feature,coefficient,abs_coefficient
0,PageValues,1.151778,1.151778
1,Month_Nov,0.453856,0.453856
2,Month_May,-0.280344,0.280344
3,ExitRates,-0.190882,0.190882
4,Month_Mar,-0.183722,0.183722
5,TotalPages,0.169246,0.169246
6,TotalDuration,0.130337,0.130337
7,BounceRates,-0.129134,0.129134
8,SpecialDay,-0.128061,0.128061
9,Month_Dec,-0.119570,0.119570


In [13]:
# Out-of-fold predictions on TRAINING data only (test set untouched)
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix

y_pred_cv = cross_val_predict(search.best_estimator_, X_train, y_train, cv=cv, n_jobs=-1)
cm = confusion_matrix(y_train, y_pred_cv)
confusion_df = pd.DataFrame(
    cm,
    index=["Actual: no purchase", "Actual: purchase"],
    columns=["Predicted: no purchase", "Predicted: purchase"],
)
confusion_df

,Predicted: no purchase,Predicted: purchase
Actual: no purchase,7357,881
Actual: purchase,323,1203


## Observations (Stage 6)
**Strength:** the model separates buyers from non-buyers well, with ROC-AUC 0.912 against 0.5 for the always-no-purchase baseline, and its coefficients are easy to interpret (PageValues is by far the strongest, then month effects and engagement features).

**Errors:** the unweighted baseline misses 45% of buyers (recall 0.547) because the data is imbalanced. Balanced weighting raised recall to about 0.80 but lowered precision, so the tuned model still flags false alarms: in out-of-fold predictions it misses 323 of 1,526 buyers and wrongly flags 881 of 8,238 non-buyers. Tuning C improved F1 from about 0.648 to 0.666, mainly through an increase in precision, while recall decreased slightly; this gain is small compared with the fold standard deviation (about 0.014).

**Feature sets:** engineered, selected, and raw feature sets all gave F1 of about 0.665 to 0.667, so feature engineering and selection made no measurable difference for this linear model.

In [14]:
import json
from joblib import dump

results_folder = project_root / "results" / "member1_logisticregression"
models_folder = project_root / "models"
results_folder.mkdir(parents=True, exist_ok=True)
models_folder.mkdir(parents=True, exist_ok=True)

baseline_results.to_csv(results_folder / "baseline.csv", index=False)
majority_results.to_csv(results_folder / "majority_class_baseline.csv", index=False)
weighting_comparison.to_csv(results_folder / "weighting_comparison.csv", index=False)
tuning_comparison.to_csv(results_folder / "tuning_comparison.csv", index=False)
feature_set_results.to_csv(results_folder / "feature_set_comparison.csv", index=False)
pagevalues_results.to_csv(results_folder / "pagevalues_comparison.csv", index=False)
experiment_summary.to_csv(results_folder / "experiment_summary.csv", index=False)
mean_std_table.to_csv(results_folder / "mean_std_summary.csv", index=False)
confusion_df.to_csv(results_folder / "cv_confusion_matrix.csv")
coef_table.to_csv(results_folder / "coefficients.csv", index=False)

# Experiments that did NOT change the decision are recorded too
pd.DataFrame({
    "C": low_search.cv_results_["param_model__C"].astype(float),
    "mean_f1": low_search.cv_results_["mean_test_f1"],
    "std_f1": low_search.cv_results_["std_test_f1"],
}).to_csv(results_folder / "c_finer_grid_check.csv", index=False)

metadata = {
    "selected_weighting": selected_weighting,
    "selected_C": float(selected_C),
    "feature_set": "X_train_selected.csv (Member 3, 18 columns)",
    "categorical_columns": categorical_cols,
    "numerical_columns": numerical_cols,
}
with open(results_folder / "candidate_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

# CANDIDATE model only - the group saves a final pipeline after comparing all algorithms
dump(search.best_estimator_, models_folder / "logisticregression_candidate.joblib")
print("Saved results to", results_folder)

Saved results to d:\FDM_Mini_Project\results\member1_logisticregression
